In [1]:
# Write your code here.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from sklearn.metrics import precision_score, recall_score, f1_score
import time
import numpy as np


In [ ]:

class DepthwiseSeparableConv(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0):
        super(DepthwiseSeparableConv, self).__init__()
        self.depthwise = nn.Conv2d(in_channels, in_channels, kernel_size, stride, padding, groups=in_channels)
        self.pointwise = nn.Conv2d(in_channels, out_channels, 1)

    def forward(self, x):
        x = self.depthwise(x)
        x = self.pointwise(x)
        return x

class DenseBlock(nn.Module):
    def __init__(self, in_channels, growth_rate, num_layers):
        super(DenseBlock, self).__init__()
        self.layers = nn.ModuleList()
        for i in range(num_layers):
            self.layers.append(nn.Sequential(
                DepthwiseSeparableConv(in_channels + i * growth_rate, growth_rate, kernel_size=3, padding=1),
                nn.BatchNorm2d(growth_rate),
                nn.ReLU(inplace=True)
            ))

    def forward(self, x):
        for layer in self.layers:
            out = layer(x)
            x = torch.cat([x, out], dim=1)
        return x

class DenseNet(nn.Module):
    def __init__(self, num_blocks, growth_rate, num_classes=10):
        super(DenseNet, self).__init__()
        self.initial_conv = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True)
        )
        self.dense_blocks = nn.ModuleList()
        in_channels = 64
        for _ in range(num_blocks):
            self.dense_blocks.append(DenseBlock(in_channels, growth_rate, num_layers=4))
            in_channels += 4 * growth_rate
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(in_channels, num_classes)

    def forward(self, x):
        x = self.initial_conv(x)
        for block in self.dense_blocks:
            x = block(x)
        x = self.global_pool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

In [ ]:

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.247, 0.243, 0.261))
])

train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")

In [ ]:

model = DenseNet(num_blocks=4, growth_rate=32).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:

def train(model, train_loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    return running_loss / len(train_loader)

def evaluate(model, test_loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_loss += loss.item()

            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    accuracy = (np.array(all_preds) == np.array(all_labels)).mean()

    precision = precision_score(all_labels, all_preds, average='macro')
    recall = recall_score(all_labels, all_preds, average='macro')
    f1 = f1_score(all_labels, all_preds, average='macro')

    return running_loss / len(test_loader), accuracy, precision, recall, f1

In [ ]:

num_epochs = 100
for epoch in range(num_epochs):
    start_time = time.time()
    train_loss = train(model, train_loader, criterion, optimizer, device)
    test_loss, accuracy, precision, recall, f1 = evaluate(model, test_loader, criterion, device)
    print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {train_loss:.4f}, Test Loss: {test_loss:.4f}, "
          f"Accuracy: {accuracy:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}, "
          f"Time: {time.time() - start_time:.2f}s")

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Number of parameters in the model: {count_parameters(model)}")

Epoch 1/100, Train Loss: 1.2155, Test Loss: 1.1915, Accuracy: 0.5892, Precision: 0.6232, Recall: 0.5892, F1: 0.5781, Time: 192.03s
Epoch 2/100, Train Loss: 0.9475, Test Loss: 0.8978, Accuracy: 0.6763, Precision: 0.6916, Recall: 0.6763, F1: 0.6616, Time: 175.42s
Epoch 3/100, Train Loss: 0.8268, Test Loss: 0.8121, Accuracy: 0.7130, Precision: 0.7314, Recall: 0.7130, F1: 0.7142, Time: 172.43s
Epoch 4/100, Train Loss: 0.7364, Test Loss: 0.8018, Accuracy: 0.7288, Precision: 0.7522, Recall: 0.7288, F1: 0.7324, Time: 172.14s
Epoch 5/100, Train Loss: 0.6608, Test Loss: 0.7520, Accuracy: 0.7379, Precision: 0.7577, Recall: 0.7379, F1: 0.7351, Time: 172.23s
Epoch 6/100, Train Loss: 0.6001, Test Loss: 0.7411, Accuracy: 0.7525, Precision: 0.7688, Recall: 0.7525, F1: 0.7509, Time: 172.25s
Epoch 7/100, Train Loss: 0.5493, Test Loss: 0.6184, Accuracy: 0.7935, Precision: 0.8062, Recall: 0.7935, F1: 0.7905, Time: 172.23s
Epoch 8/100, Train Loss: 0.5007, Test Loss: 0.5839, Accuracy: 0.7951, Precision: 0.

In [ ]:
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict()
}, "cnn_date.pth")


print("Models saved successfully!")


Models saved successfully!


In [ ]:
class StandardCNN(nn.Module):
    def __init__(self, num_classes=10):
        super(StandardCNN, self).__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.fc = nn.Linear(256 * 8 * 8, num_classes)

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x


In [ ]:
model_cnn = StandardCNN().to(device)
optimizer_cnn = optim.Adam(model_cnn.parameters(), lr=0.001)
num_epochs = 100
criterion = nn.CrossEntropyLoss()


In [ ]:
for epoch in range(num_epochs):
    start_time = time.time()
    train_loss_cnn = train(model_cnn, train_loader, criterion, optimizer_cnn, device)
    test_loss, accuracy, precision, recall, f1 = evaluate(model_cnn, test_loader, criterion, device)
    print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {train_loss_cnn:.4f}, Test Loss: {test_loss:.4f}, "
          f"Accuracy: {accuracy:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}, "
          f"Time: {time.time() - start_time:.2f}s")

Epoch 1/100, Train Loss: 0.7649, Test Loss: 0.7885, Accuracy: 0.7352, Precision: 0.7371, Recall: 0.7352, F1: 0.7316, Time: 25.77s
Epoch 2/100, Train Loss: 0.6098, Test Loss: 0.7672, Accuracy: 0.7474, Precision: 0.7496, Recall: 0.7474, F1: 0.7428, Time: 24.30s
Epoch 3/100, Train Loss: 0.4987, Test Loss: 0.7982, Accuracy: 0.7408, Precision: 0.7469, Recall: 0.7408, F1: 0.7377, Time: 25.51s
Epoch 4/100, Train Loss: 0.4005, Test Loss: 0.8553, Accuracy: 0.7429, Precision: 0.7427, Recall: 0.7429, F1: 0.7411, Time: 27.98s
Epoch 5/100, Train Loss: 0.3177, Test Loss: 0.9390, Accuracy: 0.7297, Precision: 0.7384, Recall: 0.7297, F1: 0.7320, Time: 28.84s
Epoch 6/100, Train Loss: 0.2518, Test Loss: 1.0632, Accuracy: 0.7409, Precision: 0.7434, Recall: 0.7409, F1: 0.7405, Time: 29.18s
Epoch 7/100, Train Loss: 0.2128, Test Loss: 1.1403, Accuracy: 0.7389, Precision: 0.7439, Recall: 0.7389, F1: 0.7399, Time: 29.04s
Epoch 8/100, Train Loss: 0.1768, Test Loss: 1.3225, Accuracy: 0.7296, Precision: 0.7355, R

In [ ]:

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [ ]:
# Compare Computational Efficiency
print(f"Number of parameters of DENSE net model: {count_parameters(model)}")
print(f"Number of parameters of Traditional CNN model: {count_parameters(model_cnn)}")

Number of parameters of DENSE net model: 213514
Number of parameters of Traditional CNN model: 534666
